In [124]:
!pip install  langchain langchain-core langchain-groq langchain-community sentence_transformers langchain_text_splitters pypdf chromadb faiss-cpu langchain_huggingface

In [125]:
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_community.embeddings import HuggingFaceEmbeddings

In [126]:
text_data = PyPDFLoader('/content/Psychology_Complete_Short_Guide.pdf').load()

In [127]:
for i in text_data:
  i.metadata['source'] = 'Psychology_Complete_Short_Guide'

In [240]:
splitter = RecursiveCharacterTextSplitter(chunk_size=350,chunk_overlap=50)

In [241]:
chunks = splitter.split_documents(text_data)

In [242]:
len(chunks)

8

In [244]:
from langchain_huggingface import HuggingFaceEmbeddings

# This model is 1024 dimensions.
model_name = "BAAI/bge-m3"

print(f"Downloading {model_name}... this may take a minute.")
embedd_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={'device': 'cpu'} # Use 'cuda' if you have a GPU
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [245]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedd_model)

vectorstore.save_local("rag_vector_db")

In [252]:
answer = [doc.page_content for doc in vectorstore.similarity_search("Foundations of Psychology",k=3)]

In [254]:
print(answer)

['Psychology Complete Short Guide (Beginner to\n Advanced)\n1. Foundations of Psychology\n\x7f Definition: Scientific study of mind and behavior.\n\x7f Major branches: cognitive, clinical, social, developmental, biological, industrial-organizational.\n\x7f Research methods: experiments, surveys, observation, case studies.\n2. Beginner to Advanced Concepts', '2. Beginner to Advanced Concepts\n\x7f Learning: classical conditioning, operant conditioning, observational learning.\n\x7f Memory: encoding, storage, retrieval.\n\x7f Emotion & motivation: drives, goals, rewards.\n\x7f Personality: Big Five traits, psychoanalytic, humanistic.\n3. Human Psychology Tricks & Behavior\n\x7f Reciprocity: people often return favors.', '\x7f Reciprocity: people often return favors.\n\x7f Social proof: people follow groups.\n\x7f Framing effect: wording changes decisions.\n\x7f Commitment bias: people stick with prior choices.\n4. Clinical Psychology / Mental Disorders\n\x7f Anxiety disorders: excessive 

In [257]:
llm = ChatGroq(groq_api_key="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",model="llama-3.1-8b-instant")


In [273]:
def rag():
    quiery = input("Enter Your Quiry")

    context_sematic = vectorstore.similarity_search(quiery,k=4)

    context_sematic = [i.page_content for i in context_sematic]

    context = "\n".join(context_sematic)
    prompt = f"""
    You are an intelligent RAG assistant.

    User Query:
    {quiery}

    Retrieved Context:
    {context}

    Instructions:
    1. Carefully understand the user's query.
    2. Use ONLY the retrieved context to generate the answer.
    3. If the context contains the answer, respond clearly, accurately, and naturally.
    4. If multiple relevant points exist, combine them into one well-structured response.
    5. If the context does not contain enough information, say:
      "I couldn't find enough relevant information in the provided document."
    6. Do not make up facts or use outside knowledge.
    7. Keep the response concise but helpful.
    8. If the user asks for steps, explain in numbered format.
    9. If the user asks for comparison, use table format when useful.

    Final Answer:
    """
    print(llm.invoke(prompt).content)

In [274]:
rag()

Enter Your Quiryexplain Custom Topic: Career & Success Psychology
To explain Custom Topic: Career & Success Psychology based on the retrieved context, I'll combine relevant points into a well-structured response.

Career & Success Psychology primarily focuses on understanding the psychological factors that influence career choices, success, and overall well-being. It encompasses various concepts and strategies that can be applied to achieve personal and professional growth.

Some key takeaways from the context are:

- **Growth Mindset**: Believing that skills can be improved with effort is essential for career growth and success.
- **Implementation Intentions**: Setting specific plans and intentions (e.g., "If I get a new project, then I will allocate extra time for research and planning") can help achieve goals.
- **Delayed Gratification**: Prioritizing long-term goals over short-term gains is crucial for success, as it allows for sustained effort and progress.
- **Identity-Based Habi

In [272]:
vectorstore.save_local("faiss_memory")